# Report Workflow — anti-hallucination gate in 2 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/0Smallcat0/report-workflow/blob/master/docs/quickstart_demo.ipynb)

An LLM drafts; a **deterministic** layer decides what is allowed to ship.
This notebook installs the package and runs the exact factuality checkers the
pipeline uses — no LLM, no API key, no network beyond `pip install`.

Repository: <https://github.com/0Smallcat0/report-workflow>

In [ ]:
%pip install -q "git+https://github.com/0Smallcat0/report-workflow"

## 1. Point it at ground truth

Plain strings — no schema. (In the full pipeline this evidence ledger is
built deterministically from your sources: text, CSV, PDF, DOCX.)

In [ ]:
from report_workflow import verify

SOURCES = {
    "time": "Median processing time was 12.4 minutes for the manual "
    "baseline and 7.8 minutes for the structured workflow.",
    "error": "The error rate fell to 3.5% under the structured workflow, "
    "down from 9.0% for the manual baseline.",
}

## 2. An honest draft passes every gate

In [ ]:
verify("The structured workflow cut median processing time to 7.8 minutes.", SOURCES)

## 3. A hallucinated draft is hard-blocked with the gate and reason

Two failure modes an LLM ships silently: an **invented statistic** that cites
real evidence, and a **fabricated citation** to a source that does not exist.

In [ ]:
answer = (
    "The structured workflow drove the error rate down to just 0.2% [error]. "
    "An independent third party audited and certified the results [audit_2026]."
)

result = verify(answer, SOURCES)
for row in result["sentence_results"]:
    print(f"[{row['status'].upper():8}] {row['sentence']}")
    if row["status"] != "verified":
        print(f"           gate={row['checker']}  reason: {row['reason']}")
print("publishable:", result["publishable"])

## What you just ran

- **FA** — claim/evidence/sentence linkage; rejects fabricated citations.
- **FB** — statistical claims need quantitative evidence.
- **FE** — deep audit: claim content vs evidence content (invented numbers, inflated precision, wrong units, fabricated quotes, cross-language laundering).
- **FD** — wording strength must match evidence grade.

Measured behavior (recall, false positives, documented evasions):
[`benchmarks/evidence/adversarial_2026-07-14/summary.md`](https://github.com/0Smallcat0/report-workflow/blob/master/benchmarks/evidence/adversarial_2026-07-14/summary.md).
Full pipeline (sources → evidence ledger → agent authoring → validation → DOCX):
see the [README](https://github.com/0Smallcat0/report-workflow#readme).